<a href="https://colab.research.google.com/github/viviantram03/labb-1/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install textstat

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
import textstat




# Data Preparation & Preprocessing


In [ ]:
def preprocess_isot(df, is_fake):

  df = df.dropna(subset=['text'])

  df['text'] = df['text'].astype(str)

  def clean_text(text):
    if not is_fake:
      text = re.sub(r'^.*?\(Reuters\)\s*-\s*', '', text)
    return text.strip()

  df['text'] = df['text'].apply(clean_text)

  return df[df['text'].str.strip() != ""]

print("Loading and cleaning ISOT dataset...")
true_df = pd.read_csv("True.csv", quoting=3, on_bad_lines='skip')
fake_df = pd.read_csv("Fake.csv", quoting=3, on_bad_lines='skip')

true_df = preprocess_isot(true_df, is_fake=False)
fake_df = preprocess_isot(fake_df, is_fake=True)

true_df['label'] = 0
fake_df['label'] = 1

df = pd.concat([true_df, fake_df]).sample(frac=1, random_state=42).reset_index(drop=True)



# Feature Engineering

In [ ]:
def extract_linguistic_features(texts):
  features = []
  for text in texts:
    features.append([
        textstat.flesch_reading_ease(text),
        textstat.smog_index(text),
        len(text.split()),
        text.count('!'),
        sum(1 for c in text if c.isupper()) / len(text) if len(text) > 0 else 0
    ])
  return np.array(features)

print("Extracting engineered linguistic features...")
ling_features = extract_linguistic_features(df['text'])

X_train_text, X_test_text, y_train, y_test, X_train_ling, X_test_ling = train_test_split(
    df['text'], df['label'], ling_features, test_size=0.2, random_state=42, stratify=df['label']
)

# Baseline

In [ ]:
print("Training XGBoost Baseline...")
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train_text).toarray()
X_test_tfidf = tfidf.transform(X_test_text).toarray()

X_train_xgb = np.hstack((X_train_tfidf, X_train_ling))
X_test_xgb = np.hstack((X_test_tfidf, X_test_ling))

xgb_model = XGBClassifier(n_estimators=100, learning_rate = 0.1, max_depth=6, random_state=42)
xgb_model.fit(X_train_xgb, y_train)


# Advanced: RoBERTa Fine-tuning

In [ ]:
print("Fine-tuning RoBERTa (Transfer Learning)...")
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

class FakeNewsDataset(Dataset):
  def __init__(self, texts, labels, tokenizer, max_len=256):
    self.encodings = tokenizer(texts.tolist(), truncation=True, padding='max_length', max_length=max_len)
    self.labels = labels.tolist()

  def __getitem__(self, idx):
    item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
    item['labels'] = torch.tensor(self.labels[idx])
    return item

  def __len__(self):
    return len(self.labels)

train_dataset = FakeNewsDataset(X_train_text, y_train, tokenizer)
test_dataset = FakeNewsDataset(X_test_text, y_test, tokenizer)

roberta_model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    weight_decay=0.01,
)

trainer = Trainer(
    model=roberta_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)

trainer.train()

# Hybrid Ensemble

In [ ]:
print("Implementing Hybrid Ensemble...")
xgb_probs = xgb_model.predict_proba(X_test_xgb)[:, 1]
roberta_raw = trainer.predict(test_dataset)
roberta_probs = torch.nn.functional.softmax(torch.tensor(roberta_raw.predictions), dim=1)[:, 1].numpy()

hybrid_probs = (xgb_probs + roberta_probs) / 2
y_pred_hybrid = (hybrid_probs > 0.5).astype(int)



# Evaluation & XAI (SHAP)

In [ ]:
print("Final Results (Hybrid): ")
print(classification_report(y_test, y_pred_hybrid))

print("Generating SHAP feautre importance...")
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_xgb[:100])
shap.summary_plot(shap_values, X_test_xgb[:100], feature_names=list(tfidf.get_feature_names_out()) + ['Ease', 'SMOG', 'Words', '!', 'Caps'])
